# 06 - Alarmas de calidad de datos (flags) - telemetría fija y CropX

Validaciones de `ProcessingModule.py` (flags 1 a 6), reporte por finca y etapa, panel interactivo y
exportación a Excel (`{finca}_{base}_alarmas.xlsx` en `DIR_ALARMAS`).
Umbrales en `config.py`: `valid_ranges`, `max_change`, `variables_mapping`.

> Ejecutar las celdas en orden. Usa `config.py` y `funciones_comunes.py` (misma carpeta).

In [4]:
# =============================================================================
# LIBRERÍAS
# =============================================================================
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
import ipywidgets as widgets
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
# Gradientes de color sobre filas sin datos válidos (sensor descartado): aviso inofensivo de pandas
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [5]:
# =============================================================================
# CONFIGURACION Y FUNCIONES COMUNES (editar config.py y reiniciar el kernel)
# =============================================================================
from config import *
from funciones_comunes import (_limites_rango, obtener_rangos, periodo_finca, filtrar_etapa, columnas_analisis,
                               cargar_finca, cargar_bd_fija, cargar_cropx, cargar_bd_cropx,
                               mostrar_sin_datos, contar_por_banda, _ajustar_slider)

In [6]:
# Telemetría fija y CropX de todas las fincas, recortadas al periodo de sus etapas (incluye Descanso)
BD_FIJA_FILTERED = cargar_bd_fija()
BD_CROPX = cargar_bd_cropx()
BASES = {NOMBRE_BD_FIJA: BD_FIJA_FILTERED, NOMBRE_BD_CROPX: BD_CROPX}


Cargando datos de la finca: La_Orquidea
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para La_Orquidea: 17447 filas y 43 columnas.

Cargando datos de la finca: Venecia
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Venecia: 17199 filas y 43 columnas.

Cargando datos de la finca: Puerto_Arturo
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Puerto_Arturo: 17064 filas y 43 columnas.
La_Orquidea: 8736 registros CropX, 2025-10-21 01:00:00 a 2026-10-19 12:30:00
Venecia: 8736 registros CropX, 2025-10-21 01:00:00 a 2026-10-19 12:30:00
Puerto_Arturo: 3360 registros CropX, 2026-06-02 01:00:00 a 2026-10-19 12:30:00


# FLAGS

In [7]:
# =========================================================
# ALARMAS DE CALIDAD DE DATOS (misma lógica que ProcessingModule.py)
# =========================================================
from sklearn.ensemble import IsolationForest

COLUMNAS_REPORTE = ["Fecha", "Variable", "Valor", "Flag", "Alarma", "Detalle"]
# =========================================================
# 0. FUNCIONES AUXILIARES DEL REPORTE
# =========================================================

def agregar_alarmas(reporte: list, mask: pd.Series, variable: str, valores: pd.Series,
                    flag: int, detalle) -> list:
    """Agrega al reporte una fila por cada dato marcado en `mask`.

    Args:
        reporte (list): Lista de DataFrames con las alarmas acumuladas.
        mask (pd.Series): True en los registros con alarma (índice temporal).
        variable (str): Nombre de la variable (o "Multivariable").
        valores (pd.Series): Valor que se guarda en la columna "Valor".
        flag (int): Código de la alarma (clave de FLAGS).
        detalle (str | pd.Series): Texto que explica la alarma (uno fijo o uno por registro).

    Returns:
        list: El mismo reporte con las nuevas filas.
    """
    if not mask.any():
        return reporte

    reporte.append(pd.DataFrame({
        "Fecha": mask.index[mask],
        "Variable": variable,
        "Valor": valores[mask].to_numpy(),
        "Flag": flag,
        "Alarma": FLAGS[flag],
        "Detalle": detalle[mask].to_numpy() if isinstance(detalle, pd.Series) else detalle,
    }))
    return reporte


def consolidar_reporte(reporte: list) -> pd.DataFrame:
    """Une la lista de alarmas en un solo DataFrame.

    Args:
        reporte (list): Lista de DataFrames con alarmas.

    Returns:
        pd.DataFrame: Alarmas unidas (vacío con las columnas del reporte si no hay alarmas).
    """
    if reporte:
        return pd.concat(reporte, ignore_index=True)
    return pd.DataFrame(columns=COLUMNAS_REPORTE)


# =========================================================
# 1. VALIDACION DE RANGOS (flag 1: física, flag 2: histórica)
# =========================================================

def validar_rangos(df: pd.DataFrame, variable_map: dict, valid_ranges: dict,
                   lower_percentile: float = 1, upper_percentile: float = 99) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Marca valores fuera del rango físico del sensor y fuera de los percentiles históricos.

    Args:
        df (pd.DataFrame): Telemetría de UNA finca con índice temporal.
        variable_map (dict): {columna: tipo de variable}, p. ej. variables_mapping.
        valid_ranges (dict): {tipo de variable: (mínimo, máximo)} físicos.
        lower_percentile (float): Percentil inferior histórico.
        upper_percentile (float): Percentil superior histórico.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: df con columnas `<variable>_flag` y reporte de alarmas.
    """
    df = df.copy()
    reporte = []

    for column, variable in variable_map.items():
        if column not in df.columns:
            continue
        df[f"{column}_flag"] = 0
        minimo, maximo = valid_ranges[variable]

        # 1. Anomalía física: fuera de lo que el sensor puede medir
        mask_fisica = df[column].notna() & ((df[column] < minimo) | (df[column] > maximo))
        df.loc[mask_fisica, f"{column}_flag"] = 1
        reporte = agregar_alarmas(reporte, mask_fisica, column, df[column], flag=1,
                                  detalle=f"Fuera del rango físico [{minimo}, {maximo}]")

        # 2. Anomalía histórica: fuera de P1-P99 de los datos válidos de la finca
        datos_validos = df.loc[df[f"{column}_flag"] == 0, column].dropna()
        if datos_validos.empty:
            continue
        q_low = datos_validos.quantile(lower_percentile / 100)
        q_high = datos_validos.quantile(upper_percentile / 100)

        mask_hist = (df[f"{column}_flag"] == 0) & df[column].notna() & \
                    ((df[column] < q_low) | (df[column] > q_high))
        df.loc[mask_hist, f"{column}_flag"] = 2
        reporte = agregar_alarmas(reporte, mask_hist, column, df[column], flag=2,
                                  detalle=f"Fuera de P{lower_percentile}-P{upper_percentile} "
                                          f"[{q_low:.2f}, {q_high:.2f}]")

    return df, consolidar_reporte(reporte)


# =========================================================
# 2. DETECCION DE PICOS Y VALLES (flag 3: spike aislado)
# =========================================================

def detectar_spikes(df: pd.DataFrame, variable_map: dict, max_change: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Marca un dato que salta respecto a sus dos vecinos mientras los vecinos son parecidos entre sí.

    Args:
        df (pd.DataFrame): Salida de `validar_rangos` (con columnas `_flag`).
        variable_map (dict): {columna: tipo de variable}.
        max_change (dict): {tipo de variable: cambio máximo permitido entre registros}.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: df con flags actualizados y reporte de alarmas.
    """
    df = df.copy()
    reporte = []

    for column, variable in variable_map.items():
        if f"{column}_flag" not in df.columns:
            continue
        threshold = max_change[variable]

        prev = df[column].shift(1)
        curr = df[column]
        nxt = df[column].shift(-1)

        delta_prev = (curr - prev).abs()
        delta_next = (nxt - curr).abs()
        delta_neighbors = (nxt - prev).abs()

        mask = ((df[f"{column}_flag"] == 0) & prev.notna() & curr.notna() & nxt.notna() &
                (delta_prev > threshold) & (delta_next > threshold) & (delta_neighbors <= threshold))

        df.loc[mask, f"{column}_flag"] = 3
        detalle = ("Salto de " + delta_prev.round(2).astype(str) + " y " +
                   delta_next.round(2).astype(str) + f" respecto a vecinos (umbral {threshold})")
        reporte = agregar_alarmas(reporte, mask, column, df[column], flag=3, detalle=detalle)

    return df, consolidar_reporte(reporte)


# =========================================================
# 3. VALIDACION DE CAMBIO TEMPORAL (flag 4: cambio brusco)
# =========================================================

def validar_cambio_brusco(df: pd.DataFrame, variable_map: dict, max_change: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Marca datos cuyo cambio respecto al registro anterior supera el umbral.

    Args:
        df (pd.DataFrame): Salida de `detectar_spikes`.
        variable_map (dict): {columna: tipo de variable}.
        max_change (dict): {tipo de variable: cambio máximo permitido entre registros}.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: df con flags actualizados y reporte de alarmas.
    """
    df = df.copy()
    reporte = []

    for column, variable in variable_map.items():
        if f"{column}_flag" not in df.columns:
            continue
        threshold = max_change[variable]
        delta = df[column].diff().abs()

        mask = (df[f"{column}_flag"] == 0) & df[column].notna() & (delta > threshold)

        df.loc[mask, f"{column}_flag"] = 4
        detalle = "Cambio de " + delta.round(2).astype(str) + f" (umbral {threshold})"
        reporte = agregar_alarmas(reporte, mask, column, df[column], flag=4, detalle=detalle)

    return df, consolidar_reporte(reporte)


# =========================================================
# 4. OUTLIERS LOCALES - ROLLING MAD (flag 5)
# =========================================================

def validar_outliers_mad(df: pd.DataFrame, variable_map: dict,
                         window: int = 48, threshold: float = 4) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Marca datos alejados de la mediana móvil según un z-score robusto (MAD).

    Args:
        df (pd.DataFrame): Salida de `validar_cambio_brusco`.
        variable_map (dict): {columna: tipo de variable}.
        window (int): Tamaño de la ventana móvil en registros (48 = ~24 h con datos cada 30 min).
        threshold (float): Z-score robusto a partir del cual se marca la alarma.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: df con flags actualizados y reporte de alarmas.
    """
    df = df.copy()
    reporte = []

    for column in variable_map:
        if f"{column}_flag" not in df.columns:
            continue

        # Excluir errores claros (físicos y spikes) del cálculo
        serie = df[column].where(~df[f"{column}_flag"].isin([1, 3]))

        rolling_median = serie.rolling(window, center=True, min_periods=window // 2).median()
        rolling_mad = (serie - rolling_median).abs().rolling(
            window, center=True, min_periods=window // 2).median()

        rolling_mad = rolling_mad.where(rolling_mad > 1e-9)   # MAD = 0 (p. ej. radiación de noche) no se evalúa
        robust_score = (serie - rolling_median).abs() / (1.4826 * rolling_mad)

        mask = df[column].notna() & (df[f"{column}_flag"] == 0) & (robust_score > threshold)

        df.loc[mask, f"{column}_flag"] = 5
        detalle = "Z-score robusto " + robust_score.round(1).astype(str) + f" (umbral {threshold})"
        reporte = agregar_alarmas(reporte, mask, column, df[column], flag=5, detalle=detalle)

    return df, consolidar_reporte(reporte)


# =========================================================
# 5. ANOMALIAS MULTIVARIABLES - ISOLATION FOREST (flag 6)
# =========================================================

def detectar_anomalias_multivariables(df: pd.DataFrame, variables: list[str], contamination: float = 0.01,
                                      random_state: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Marca registros cuya combinación de variables es atípica (Isolation Forest de la finca).

    Args:
        df (pd.DataFrame): Salida de `validar_outliers_mad`.
        variables (list[str]): Columnas que usa el modelo.
        contamination (float): Proporción esperada de registros anómalos.
        random_state (int): Semilla para resultados reproducibles.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: df con columna `multivariable_flag` y reporte de alarmas.
    """
    df = df.copy()
    df["multivariable_flag"] = 0
    if len(df) < 2 or not variables:                     # sin datos suficientes: no se evalúa
        return df, consolidar_reporte([])

    X = df[variables].fillna(df[variables].median())     # Isolation Forest no acepta NaN
    model = IsolationForest(contamination=contamination, random_state=random_state,
                            n_estimators=200, n_jobs=-1).fit(X)

    score = pd.Series(model.decision_function(X), index=df.index)
    mask = pd.Series(model.predict(X) == -1, index=df.index)

    df.loc[mask, "multivariable_flag"] = 6
    detalle = "Puntaje Isolation Forest " + score.round(3).astype(str)
    reporte = agregar_alarmas([], mask, "Multivariable", score, flag=6, detalle=detalle)

    return df, consolidar_reporte(reporte)


# =========================================================
# 6. PIPELINE DE ALARMAS POR FINCA
# =========================================================

def asignar_etapa(df: pd.DataFrame, finca: str) -> pd.Series:
    """Nombre de la etapa a la que pertenece cada registro.

    Args:
        df (pd.DataFrame): Telemetría de la finca con índice temporal.
        finca (str): Nombre de la finca.

    Returns:
        pd.Series: Etapa de cada registro ("Sin etapa" si no cae en ninguna).
    """
    etapa = pd.Series("Sin etapa", index=df.index, name="Etapa")
    for nombre in obtener_rangos(FINCAS_POR_NOMBRE[finca]):
        etapa.loc[filtrar_etapa(df, finca, nombre).index] = nombre
    return etapa


def generar_alarmas_finca(df: pd.DataFrame, finca: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Aplica todas las validaciones en orden. Cada dato conserva la PRIMERA alarma que recibe.

    Args:
        df (pd.DataFrame): Telemetría de la finca con índice temporal.
        finca (str): Nombre de la finca.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: df con columnas de flags y "Etapa", y reporte de alarmas.
    """
    variables_if = [c for c in columnas_analisis(df)
                    if not c.endswith("_pred") and not c.startswith("conductivity_est")]

    df, r1 = validar_rangos(df, variables_mapping, valid_ranges)
    df, r2 = detectar_spikes(df, variables_mapping, max_change)
    df, r3 = validar_cambio_brusco(df, variables_mapping, max_change)
    df, r4 = validar_outliers_mad(df, variables_mapping, window=48, threshold=4)
    df, r5 = detectar_anomalias_multivariables(df, variables_if)

    df["Etapa"] = asignar_etapa(df, finca)
    reporte = consolidar_reporte([r for r in (r1, r2, r3, r4, r5) if not r.empty])
    reporte.insert(0, "Finca", finca)
    reporte.insert(1, "Etapa", df["Etapa"].reindex(reporte["Fecha"]).to_numpy())
    return df, reporte.sort_values(["Fecha", "Flag"]).reset_index(drop=True)


# =========================================================
# 7. RESUMEN Y VISUALIZACION DE ALARMAS
# =========================================================

def resumen_alarmas(reporte: pd.DataFrame) -> pd.DataFrame:
    """Resume las alarmas por finca, etapa, variable y tipo, con sus fechas.

    Args:
        reporte (pd.DataFrame): Reporte de alarmas (una o varias fincas).

    Returns:
        pd.DataFrame: Cantidad de alarmas, días con alarma y primera/última fecha.
    """
    reporte = reporte.sort_values(["Finca", "Fecha"])
    return (reporte
            .groupby(["Finca", "Etapa", "Variable", "Alarma"], sort=False)
            .agg(Alarmas=("Fecha", "size"),
                 Dias_con_alarma=("Fecha", lambda f: f.dt.normalize().nunique()),
                 Primera_fecha=("Fecha", "min"),
                 Ultima_fecha=("Fecha", "max"))
            .rename(columns={"Dias_con_alarma": "Días con alarma",
                             "Primera_fecha": "Primera fecha",
                             "Ultima_fecha": "Última fecha"}))


def graficar_alarmas(df: pd.DataFrame, finca: str, variable: str, flags: list[int]) -> None:
    """Serie temporal de la variable con las alarmas marcadas por color y las etapas sombreadas.

    Args:
        df (pd.DataFrame): Salida de `generar_alarmas_finca` (puede estar recortada a una etapa).
        finca (str): Texto que identifica finca, base y etapa en el título.
        variable (str): Variable a graficar.
        flags (list[int]): Códigos de alarma a mostrar (1 a 6).

    Returns:
        None: Muestra la figura.
    """
    if df.empty or df[variable].dropna().empty:
        mostrar_sin_datos(f"Alarmas de {variable} - {finca}", figsize=(15, 3))
        return

    fig, ax = plt.subplots(figsize=(15, 5))
    linea = df[variable].resample("30min").mean()          # los huecos sin datos quedan en blanco
    ax.plot(linea.index, linea, color="0.65", linewidth=0.7, label="Dato")

    for flag in flags:
        columna_flag = "multivariable_flag" if flag == 6 else f"{variable}_flag"
        mask = df[columna_flag] == flag
        if mask.any():
            ax.scatter(df.index[mask], df.loc[mask, variable], s=16, zorder=3,
                       color=COLORES_FLAGS[flag], marker="x" if flag == 6 else "o",
                       label=f"{flag} - {FLAGS[flag]} ({mask.sum()})")

    for i, (etapa, tramo) in enumerate(df.groupby("Etapa", sort=False)):
        ax.axvspan(tramo.index.min(), tramo.index.max(), color=f"C{i}", alpha=0.07)
        ax.text(tramo.index.min(), 1.01 + 0.05 * (i % 2), etapa,   # alterna altura para que no se monten
                transform=ax.get_xaxis_transform(), fontsize=9)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%y"))
    ax.set_ylabel(variable)
    ax.set_title(f"Alarmas de {variable} - {finca}", pad=28)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    plt.tight_layout()
    plt.show()


def calcular_alarmas(bd: dict[str, pd.DataFrame]) -> tuple[dict, dict, pd.DataFrame]:
    """Aplica `generar_alarmas_finca` a todas las fincas de una base.

    Args:
        bd (dict[str, pd.DataFrame]): {finca: datos con índice temporal} (BD_FIJA_FILTERED o BD_CROPX).

    Returns:
        tuple[dict, dict, pd.DataFrame]: {finca: df con flags y Etapa}, {finca: reporte} y reporte de todas las fincas.
    """
    flags_df, reportes = {}, {}
    for finca, df in bd.items():
        flags_df[finca], reportes[finca] = generar_alarmas_finca(df, finca)
    total = pd.concat(reportes.values(), ignore_index=True) if reportes else consolidar_reporte([])
    return flags_df, reportes, total


def guardar_alarmas_excel(reportes: dict[str, pd.DataFrame], nombre_bd: str) -> list[Path]:
    """Guarda un Excel por finca: {finca}_{nombre_bd}_alarmas.xlsx con hojas Resumen y Detalle.

    Args:
        reportes (dict[str, pd.DataFrame]): {finca: reporte de alarmas} (salida 2 de `calcular_alarmas`).
        nombre_bd (str): Nombre de la base (NOMBRE_BD_FIJA o NOMBRE_BD_CROPX).

    Returns:
        list[Path]: Rutas de los archivos guardados.
    """
    DIR_ALARMAS.mkdir(parents=True, exist_ok=True)
    rutas = []
    for finca, reporte in reportes.items():
        ruta = DIR_ALARMAS / f"{finca}_{nombre_bd}_alarmas.xlsx"
        resumen = resumen_alarmas(reporte).reset_index() if not reporte.empty else pd.DataFrame(
            columns=["Finca", "Etapa", "Variable", "Alarma", "Alarmas", "Días con alarma",
                     "Primera fecha", "Última fecha"])
        with pd.ExcelWriter(ruta) as excel:
            resumen.to_excel(excel, sheet_name="Resumen", index=False)
            reporte.to_excel(excel, sheet_name="Detalle", index=False)
        rutas.append(ruta)
        print(f"Guardado: {ruta} ({len(reporte)} alarmas)")
    return rutas


def panel_alarmas(flags_df: dict[str, pd.DataFrame], reporte_total: pd.DataFrame, nombre_bd: str) -> None:
    """Widget: elige finca, variable, etapa y tipos de alarma; muestra gráfica y resumen.

    Args:
        flags_df (dict[str, pd.DataFrame]): {finca: df con flags y Etapa} (salida 1 de `calcular_alarmas`).
        reporte_total (pd.DataFrame): Reporte de todas las fincas (salida 3 de `calcular_alarmas`).
        nombre_bd (str): Nombre de la base para los títulos.

    Returns:
        None: Muestra los widgets, la gráfica y la tabla de alarmas filtrada.
    """
    if not flags_df:
        mostrar_sin_datos(f"Alarmas - {nombre_bd}", "No hay fincas cargadas en esta base")
        return
    w_finca = widgets.Dropdown(options=list(flags_df), description="Finca")
    w_variable = widgets.Dropdown(description="Variable")
    w_etapa = widgets.Dropdown(description="Etapa")
    w_flags = widgets.SelectMultiple(options=[(f"{k} - {v}", k) for k, v in FLAGS.items() if 1 <= k <= 6],
                                     value=tuple(range(1, 7)), rows=6, description="Alarmas")

    def al_cambiar_finca(cambio=None):
        df = flags_df[w_finca.value]
        w_variable.options = [c for c in variables_mapping
                              if f"{c}_flag" in df.columns and df[c].notna().any()]
        w_etapa.options = ["Todo el periodo", "Ciclo completo", *obtener_rangos(FINCAS_POR_NOMBRE[w_finca.value])]
        w_variable.value = w_variable.options[0] if w_variable.options else None   # ipywidgets 8 no elige solo
        w_etapa.value = "Todo el periodo"

    def actualizar(finca, variable, etapa, flags):
        if etapa is None:
            return
        if not variable:
            mostrar_sin_datos(f"Alarmas - {nombre_bd} - {finca}", "La finca no tiene variables con datos")
            return
        df = flags_df[finca]
        if etapa != "Todo el periodo":
            df = filtrar_etapa(df, finca, None if etapa == "Ciclo completo" else etapa)
        graficar_alarmas(df, f"{nombre_bd} - {finca} - {etapa}", variable, list(flags))

        tabla = reporte_total[(reporte_total["Finca"] == finca) &
                              (reporte_total["Variable"].isin([variable, "Multivariable"])) &
                              (reporte_total["Flag"].isin(flags))]
        if etapa != "Todo el periodo" and not df.empty:
            tabla = tabla[tabla["Fecha"].isin(df.index)]
        elif etapa != "Todo el periodo":
            tabla = tabla.iloc[0:0]
        display(resumen_alarmas(tabla) if not tabla.empty else Markdown("_Sin alarmas con estos filtros._"))

    al_cambiar_finca()
    w_finca.observe(al_cambiar_finca, "value")
    salida = widgets.interactive_output(actualizar, {"finca": w_finca, "variable": w_variable,
                                                     "etapa": w_etapa, "flags": w_flags})
    display(widgets.HBox([widgets.VBox([w_finca, w_variable, w_etapa]), w_flags]), salida)

## Telemetría fija

In [8]:
FLAGS_ALARMAS, REPORTES, REPORTE_ALARMAS = calcular_alarmas(BD_FIJA_FILTERED)
guardar_alarmas_excel(REPORTES, NOMBRE_BD_FIJA)
REPORTE_ALARMAS.head(20)

Guardado: D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Code\Data_Exploration\EDA_telemetria_v2\outputs\EDA_telemetria_fija\alarmas\La_Orquidea_telemetria_fija_alarmas.xlsx (20138 alarmas)
Guardado: D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Code\Data_Exploration\EDA_telemetria_v2\outputs\EDA_telemetria_fija\alarmas\Venecia_telemetria_fija_alarmas.xlsx (11480 alarmas)
Guardado: D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Code\Data_Exploration\EDA_telemetria_v2\outputs\EDA_telemetria_fija\alarmas\Puerto_Arturo_telemetria_fija_alarmas.xlsx (19413 alarmas)


,Finca,Etapa,Fecha,Variable,Valor,Flag,Alarma,Detalle
0,La_Orquidea,Descanso,2025-10-21 09:23:54,medium_moisture_40cm,0.00,2,Anomalía histórica,"Fuera de P1-P99 [9.50, 32.40]"
1,La_Orquidea,Descanso,2025-10-21 09:23:54,bottom_moisture_60cm,0.00,2,Anomalía histórica,"Fuera de P1-P99 [2.60, 63.10]"
2,La_Orquidea,Descanso,2025-10-21 09:23:54,top_conductivity_20cm,0.00,5,Outlier MAD,Z-score robusto 35.1 (umbral 4)
3,La_Orquidea,Descanso,2025-10-21 09:23:54,medium_conductivity_40cm,0.00,5,Outlier MAD,Z-score robusto 5.9 (umbral 4)
4,La_Orquidea,Descanso,2025-10-21 09:23:54,bottom_conductivity_60cm,0.00,5,Outlier MAD,Z-score robusto 7.8 (umbral 4)
5,La_Orquidea,Descanso,2025-10-21 09:23:54,medium_ph_40cm,7.00,5,Outlier MAD,Z-score robusto 8.9 (umbral 4)
6,La_Orquidea,Descanso,2025-10-21 09:23:54,top_temperature_20cm,20.80,5,Outlier MAD,Z-score robusto 4.0 (umbral 4)
7,La_Orquidea,Descanso,2025-10-21 09:23:54,top_moisture_20cm,0.00,5,Outlier MAD,Z-score robusto 48.7 (umbral 4)
8,La_Orquidea,Descanso,2025-10-21 09:23:54,air_temperature,22.70,5,Outlier MAD,Z-score robusto 7.8 (umbral 4)
9,La_Orquidea,Descanso,2025-10-21 09:23:54,air_humidity,75.30,5,Outlier MAD,Z-score robusto 8.1 (umbral 4)


In [9]:
resumen_alarmas(REPORTES["La_Orquidea"]) if not REPORTES["La_Orquidea"].empty else "Sin alarmas"

Alarmas  Días con alarma       Primera fecha        Última fecha
Finca       Etapa     Variable                 Alarma                                                                              
La_Orquidea Descanso  medium_moisture_40cm     Anomalía histórica       13                2 2025-10-21 09:23:54 2025-10-22 14:28:50
                      bottom_moisture_60cm     Anomalía histórica      163                5 2025-10-21 09:23:54 2025-10-25 01:55:20
                      top_conductivity_20cm    Outlier MAD              54               19 2025-10-21 09:23:54 2025-11-18 02:10:48
                      medium_conductivity_40cm Outlier MAD               7                5 2025-10-21 09:23:54 2025-11-11 17:35:38
                      bottom_conductivity_60cm Outlier MAD              35               18 2025-10-21 09:23:54 2025-11-22 14:45:45
...                                                                    ...              ...                 ...                 ...
            Descanso2 top_moisture_20cm        Cambio brusco             1                1 2026-08-30 19:06:32 2026-08-30 19:06:32
                      medium_moisture_40cm     Cambio brusco             8                4 2026-09-03 13:53:16 2026-10-01 12:22:29
                      bottom_moisture_60cm     Cambio brusco             1                1 2026-09-03 13:53:16 2026-09-03 13:53:16
                      top_ph_20cm              Spike aislado             1                1 2026-09-09 02:10:28 2026-09-09 02:10:28
                      medium_moisture_40cm     Spike aislado             6                3 2026-09-29 11:27:07 2026-10-01 10:52:07

[195 rows x 4 columns]

In [10]:
resumen_alarmas(REPORTES["Venecia"]) if not REPORTES["Venecia"].empty else "Sin alarmas"

Alarmas  Días con alarma       Primera fecha        Última fecha
Finca   Etapa    Variable                 Alarma                                                                                  
Venecia Descanso medium_conductivity_40cm Anomalía histórica          272               49 2025-10-21 11:10:53 2026-05-03 09:06:18
                 medium_ph_40cm           Anomalía histórica           92                9 2025-10-21 11:10:53 2025-11-21 21:37:41
                 bottom_ph_60cm           Anomalía histórica           25                4 2025-10-21 11:10:53 2026-05-04 22:21:25
                 bottom_moisture_60cm     Anomalía histórica          185                9 2025-10-21 11:10:53 2026-05-04 03:12:48
                 top_conductivity_20cm    Outlier MAD                 614              156 2025-10-21 11:10:53 2026-05-04 13:17:21
...                                                                   ...              ...                 ...                 ...
        Cosecha  bottom_moisture_60cm     Anomalía histórica           89               11 2026-10-05 18:58:45 2026-10-19 10:09:04
                 bottom_temperature_60cm  Outlier MAD                   2                2 2026-10-07 23:48:44 2026-10-08 00:18:35
                 radiation                Anomalía histórica            2                2 2026-10-14 11:40:47 2026-10-17 11:16:43
                 medium_moisture_40cm     Anomalía histórica          151                6 2026-10-14 12:40:31 2026-10-19 19:09:36
                 Multivariable            Anomalía multivariable      129                5 2026-10-15 08:08:22 2026-10-19 13:39:15

[150 rows x 4 columns]

In [11]:
resumen_alarmas(REPORTES["Puerto_Arturo"]) if not REPORTES["Puerto_Arturo"].empty else "Sin alarmas"

Alarmas  Días con alarma       Primera fecha        Última fecha
Finca         Etapa     Variable                 Alarma                                                                              
Puerto_Arturo Descanso  medium_conductivity_40cm Anomalía histórica      102               17 2025-10-21 16:17:48 2025-12-07 16:35:14
                        top_ph_20cm              Anomalía histórica      166                9 2025-10-21 16:17:48 2025-12-15 20:17:18
                        medium_moisture_40cm     Anomalía histórica      104                3 2025-10-21 16:17:48 2025-10-24 21:14:19
                        bottom_moisture_60cm     Anomalía histórica       94                3 2025-10-21 16:17:48 2025-10-24 21:14:19
                        top_conductivity_20cm    Outlier MAD             118               41 2025-10-21 16:17:48 2026-01-26 15:33:25
...                                                                      ...              ...                 ...                 ...
              Descanso2 top_temperature_20cm     Spike aislado             2                2 2026-09-26 11:11:35 2026-10-09 10:15:59
                        air_humidity             Cambio brusco             3                3 2026-10-04 16:34:58 2026-10-17 16:02:23
                        medium_moisture_40cm     Anomalía histórica        1                1 2026-10-08 15:21:20 2026-10-08 15:21:20
                        medium_conductivity_40cm Anomalía histórica       16                1 2026-10-09 15:50:04 2026-10-09 23:20:01
                        bottom_conductivity_60cm Anomalía histórica       78                4 2026-10-15 11:05:46 2026-10-18 21:00:16

[181 rows x 4 columns]

In [12]:
panel_alarmas(FLAGS_ALARMAS, REPORTE_ALARMAS, NOMBRE_BD_FIJA)

Output()

## CropX

In [13]:
FLAGS_ALARMAS_CROPX, REPORTES_CROPX, REPORTE_ALARMAS_CROPX = calcular_alarmas(BD_CROPX)
guardar_alarmas_excel(REPORTES_CROPX, NOMBRE_BD_CROPX)
REPORTE_ALARMAS_CROPX.head(20)

Guardado: D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Code\Data_Exploration\EDA_telemetria_v2\outputs\EDA_telemetria_fija\alarmas\La_Orquidea_cropx_alarmas.xlsx (2510 alarmas)
Guardado: D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Code\Data_Exploration\EDA_telemetria_v2\outputs\EDA_telemetria_fija\alarmas\Venecia_cropx_alarmas.xlsx (2171 alarmas)
Guardado: D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Code\Data_Exploration\EDA_telemetria_v2\outputs\EDA_telemetria_fija\alarmas\Puerto_Arturo_cropx_alarmas.xlsx (2497 alarmas)


,Finca,Etapa,Fecha,Variable,Valor,Flag,Alarma,Detalle
0,La_Orquidea,Descanso,2025-10-21 10:30:00,ECpw 20,1.544,2,Anomalía histórica,"Fuera de P1-P99 [0.18, 0.44]"
1,La_Orquidea,Descanso,2025-10-21 10:30:00,ECpw 41,0.088,2,Anomalía histórica,"Fuera de P1-P99 [0.13, 0.42]"
2,La_Orquidea,Descanso,2025-10-21 10:30:00,ECpw 66,0.081,2,Anomalía histórica,"Fuera de P1-P99 [0.13, 0.40]"
3,La_Orquidea,Descanso,2025-10-21 10:30:00,Moisture 20,44.530,5,Outlier MAD,Z-score robusto 4.1 (umbral 4)
4,La_Orquidea,Descanso,2025-10-21 10:30:00,Moisture 41,41.980,5,Outlier MAD,Z-score robusto 16.3 (umbral 4)
5,La_Orquidea,Descanso,2025-10-21 11:00:00,ECpw 20,1.498,2,Anomalía histórica,"Fuera de P1-P99 [0.18, 0.44]"
6,La_Orquidea,Descanso,2025-10-21 11:00:00,ECpw 41,0.105,2,Anomalía histórica,"Fuera de P1-P99 [0.13, 0.42]"
7,La_Orquidea,Descanso,2025-10-21 11:00:00,ECpw 66,0.094,2,Anomalía histórica,"Fuera de P1-P99 [0.13, 0.40]"
8,La_Orquidea,Descanso,2025-10-21 11:00:00,Moisture 20,44.580,5,Outlier MAD,Z-score robusto 4.1 (umbral 4)
9,La_Orquidea,Descanso,2025-10-21 11:00:00,Moisture 41,41.980,5,Outlier MAD,Z-score robusto 17.2 (umbral 4)


In [14]:
for finca, reporte in REPORTES_CROPX.items():
    display(Markdown(f"### {finca}"))
    display(resumen_alarmas(reporte) if not reporte.empty else Markdown("_Sin alarmas._"))

### La_Orquidea

Alarmas  Días con alarma       Primera fecha        Última fecha
Finca       Etapa     Variable    Alarma                                                                              
La_Orquidea Descanso  ECpw 20     Anomalía histórica        5                2 2025-10-21 10:30:00 2025-11-22 07:30:00
                      ECpw 41     Anomalía histórica       81                6 2025-10-21 10:30:00 2025-10-28 12:30:00
                      ECpw 66     Anomalía histórica       64                8 2025-10-21 10:30:00 2025-11-02 11:00:00
                      Moisture 20 Outlier MAD               7                3 2025-10-21 10:30:00 2025-11-18 12:30:00
                      Moisture 41 Outlier MAD              14                3 2025-10-21 10:30:00 2025-11-16 12:30:00
...                                                       ...              ...                 ...                 ...
            Descanso2 Moisture 20 Cambio brusco            14                6 2026-09-03 03:00:00 2026-10-16 12:00:00
                      Moisture 41 Cambio brusco             7                2 2026-09-03 03:00:00 2026-09-04 01:00:00
                      Moisture 66 Cambio brusco             4                1 2026-09-03 03:00:00 2026-09-03 12:00:00
                      ECpw 66     Anomalía histórica       88                8 2026-09-15 02:30:00 2026-10-17 12:30:00
                      ECpw 20     Cambio brusco             1                1 2026-10-18 06:30:00 2026-10-18 06:30:00

[72 rows x 4 columns]

### Venecia

Alarmas  Días con alarma       Primera fecha        Última fecha
Finca   Etapa    Variable       Alarma                                                                              
Venecia Descanso ECpw 20        Anomalía histórica       91               22 2025-10-21 01:00:00 2026-03-30 03:00:00
                 ECpw 41        Anomalía histórica      102                8 2025-10-21 01:00:00 2026-03-07 01:00:00
                 ECpw 66        Anomalía histórica       92                9 2025-10-21 01:00:00 2026-04-17 04:30:00
                 Moisture 20    Anomalía histórica       76                5 2025-10-21 01:00:00 2025-10-25 11:30:00
                 Moisture 66    Anomalía histórica       77                4 2025-10-21 01:00:00 2025-10-24 12:00:00
...                                                     ...              ...                 ...                 ...
        Cosecha  Moisture 66    Anomalía histórica       76               10 2026-10-07 04:30:00 2026-10-19 02:30:00
                 Moisture 20    Anomalía histórica       84                9 2026-10-08 05:30:00 2026-10-17 02:30:00
                 Moisture 41    Anomalía histórica       31                7 2026-10-08 10:30:00 2026-10-14 06:30:00
                 Temperature 20 Cambio brusco             6                4 2026-10-09 09:00:00 2026-10-18 12:00:00
                                Spike aislado             4                3 2026-10-14 10:00:00 2026-10-18 11:30:00

[69 rows x 4 columns]

### Puerto_Arturo

Alarmas  Días con alarma       Primera fecha        Última fecha
Finca         Etapa     Variable       Alarma                                                                                  
Puerto_Arturo Floración Moisture 20    Outlier MAD                   6                3 2026-06-02 12:00:00 2026-06-07 12:30:00
                        Moisture 41    Outlier MAD                   1                1 2026-06-08 12:00:00 2026-06-08 12:00:00
                        ECpw 66        Outlier MAD                   1                1 2026-06-10 04:30:00 2026-06-10 04:30:00
                        Temperature 20 Cambio brusco                10                3 2026-06-10 06:30:00 2026-06-12 09:00:00
                        ECpw 20        Outlier MAD                   8                3 2026-06-10 08:00:00 2026-06-14 12:30:00
                        Temperature 20 Spike aislado                 6                2 2026-06-10 09:00:00 2026-06-11 08:00:00
              Cosecha   Moisture 20    Outlier MAD                  33                9 2026-06-16 02:30:00 2026-08-18 02:00:00
                        ECpw 41        Outlier MAD                  47               12 2026-06-16 11:00:00 2026-08-30 03:30:00
                        ECpw 20        Outlier MAD                   9                4 2026-06-19 01:00:00 2026-07-15 12:30:00
                                       Anomalía física               1                1 2026-06-19 07:00:00 2026-06-19 07:00:00
                                       Cambio brusco                 1                1 2026-06-19 07:30:00 2026-06-19 07:30:00
                                       Anomalía histórica           27                5 2026-06-20 01:00:00 2026-07-16 04:00:00
                        Temperature 20 Cambio brusco               132               49 2026-06-22 09:00:00 2026-09-01 07:30:00
                        Moisture 41    Outlier MAD                   5                3 2026-06-23 12:00:00 2026-08-17 03:00:00
                        ECpw 66        Outlier MAD                  53               11 2026-06-25 07:00:00 2026-08-13 12:30:00
                        Temperature 20 Spike aislado               103               38 2026-06-27 07:30:00 2026-09-01 07:00:00
                        ECpw 41        Anomalía física               1                1 2026-06-29 07:00:00 2026-06-29 07:00:00
                        ECpw 66        Anomalía histórica            2                1 2026-06-29 07:00:00 2026-06-29 09:00:00
                        ECpw 41        Cambio brusco                 1                1 2026-06-29 07:30:00 2026-06-29 07:30:00
                        ECpw 66        Cambio brusco                 1                1 2026-06-29 07:30:00 2026-06-29 07:30:00
                        ECpw 41        Anomalía histórica           33                4 2026-06-29 09:00:00 2026-08-13 07:00:00
                        Moisture 66    Outlier MAD                  14                6 2026-07-10 08:00:00 2026-08-30 03:30:00
                        Temperature 20 Outlier MAD                   1                1 2026-07-11 12:00:00 2026-07-11 12:00:00
                        Temperature 41 Outlier MAD                   2                2 2026-08-07 05:30:00 2026-08-17 12:00:00
                        Temperature 20 Anomalía histórica            5                1 2026-08-23 05:30:00 2026-08-23 07:30:00
              Descanso2 Temperature 20 Spike aislado                70               35 2026-09-02 05:30:00 2026-10-19 08:30:00
                                       Cambio brusco               233               48 2026-09-02 06:00:00 2026-10-19 12:00:00
                                       Anomalía histórica           41               15 2026-09-04 04:30:00 2026-10-16 02:00:00
                                       Anomalía física             208               34 2026-09-04 05:30:00 2026-10-19 11:00:00
                        Temperature 41 Anomalía histórica           59               18 2026-09

In [15]:
panel_alarmas(FLAGS_ALARMAS_CROPX, REPORTE_ALARMAS_CROPX, NOMBRE_BD_CROPX)

Output()